In [ ]:
%pip install openmeteo-requests
%pip install requests-cache retry-requests numpy pandas

In [1]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

In [2]:
def get_aq_data(coordinates,start_date , end_date) -> pd.DataFrame:
    '''
        Fetchs the Air Quality data ("pm10", "pm2_5", "uv_index", "carbon_monoxide", "carbon_dioxide", "nitrogen_dioxide", "sulphur_dioxide", "ozone") 
        From "start_date" to "end_date" 
        For the corresponding "coordinates" 
        via Open-Meteo API 
    '''
    
    # Setup the Open-Meteo API client with cache and retry on error
    cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
    retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
    openmeteo = openmeteo_requests.Client(session = retry_session)

    # Make sure all required weather variables are listed here
    # The order of variables in hourly or daily is important to assign them correctly below
    url = "https://air-quality-api.open-meteo.com/v1/air-quality"
    params = {
        "latitude":coordinates['Latitude'],
        "longitude": coordinates['Longitude'],
        "hourly": ["pm10", "pm2_5", "uv_index", "carbon_monoxide", "carbon_dioxide", "nitrogen_dioxide", "sulphur_dioxide", "ozone"],
        "start_date": start_date,
        "end_date":end_date,
    }
    responses = openmeteo.weather_api(url, params = params)

    # Process first location. Add a for-loop for multiple locations or weather models
    response = responses[0]
    print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
    print(f"Elevation: {response.Elevation()} m asl")
    print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

    # Process hourly data. The order of variables needs to be the same as requested.
    hourly = response.Hourly()
    hourly_pm10 = hourly.Variables(0).ValuesAsNumpy()
    hourly_pm2_5 = hourly.Variables(1).ValuesAsNumpy()
    hourly_uv_index = hourly.Variables(2).ValuesAsNumpy()
    hourly_carbon_monoxide = hourly.Variables(3).ValuesAsNumpy()
    hourly_carbon_dioxide = hourly.Variables(4).ValuesAsNumpy()
    hourly_nitrogen_dioxide = hourly.Variables(5).ValuesAsNumpy()
    hourly_sulphur_dioxide = hourly.Variables(6).ValuesAsNumpy()
    hourly_ozone = hourly.Variables(7).ValuesAsNumpy()

    hourly_data = {
        "date": pd.date_range(
            start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
            end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
            freq = pd.Timedelta(seconds = hourly.Interval()),
            inclusive = "left"
        )
    }

    hourly_data["pm10"] = hourly_pm10
    hourly_data["pm2_5"] = hourly_pm2_5
    hourly_data["uv_index"] = hourly_uv_index
    hourly_data["carbon_monoxide"] = hourly_carbon_monoxide
    hourly_data["carbon_dioxide"] = hourly_carbon_dioxide
    hourly_data["nitrogen_dioxide"] = hourly_nitrogen_dioxide
    hourly_data["sulphur_dioxide"] = hourly_sulphur_dioxide
    hourly_data["ozone"] = hourly_ozone

    aq_df = pd.DataFrame(data = hourly_data)
    return aq_df

In [3]:
def get_weather_data(coordinates,start_date , end_date) -> pd.DataFrame:
    '''
        Fetchs the Weather data (temperature_2m", "relative_humidity_2m", "surface_pressure", "precipitation", "wind_speed_10m", "wind_direction_10m") 
        From "start_date" to "end_date" 
        For the corresponding "coordinates" 
        via Open-Meteo API 
    '''
    
    # Setup the Open-Meteo API client with cache and retry on error
    cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
    retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
    openmeteo = openmeteo_requests.Client(session = retry_session)

    # Make sure all required weather variables are listed here
    # The order of variables in hourly or daily is important to assign them correctly below
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude":coordinates['Latitude'],
        "longitude": coordinates['Longitude'],
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ["temperature_2m", "relative_humidity_2m", "surface_pressure", "precipitation", "wind_speed_10m", "wind_direction_10m"],
    }
    responses = openmeteo.weather_api(url, params = params)

    # Process first location. Add a for-loop for multiple locations or weather models
    response = responses[0]
    print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
    print(f"Elevation: {response.Elevation()} m asl")
    print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

    # Process hourly data. The order of variables needs to be the same as requested.
    hourly = response.Hourly()
    hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
    hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
    hourly_surface_pressure = hourly.Variables(2).ValuesAsNumpy()
    hourly_precipitation = hourly.Variables(3).ValuesAsNumpy()
    hourly_wind_speed_10m = hourly.Variables(4).ValuesAsNumpy()
    hourly_wind_direction_10m = hourly.Variables(5).ValuesAsNumpy()

    hourly_data = {
        "date": pd.date_range(
            start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
            end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
            freq = pd.Timedelta(seconds = hourly.Interval()),
            inclusive = "left"
        )
    }

    hourly_data["temperature_2m"] = hourly_temperature_2m
    hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
    hourly_data["surface_pressure"] = hourly_surface_pressure
    hourly_data["precipitation"] = hourly_precipitation
    hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
    hourly_data["wind_direction_10m"] = hourly_wind_direction_10m

    weather_df = pd.DataFrame(data = hourly_data)
    return weather_df

In [9]:
districts_map = {
    'Helwan (Industrial South)': {'Latitude': 29.8490, 'Longitude': 31.3342},
    'Shobra El-Kheima (Industrial North)': {'Latitude': 30.1286, 'Longitude': 31.2422},
    'New Cairo / 5th Settlement (Eastern Residential Suburbs)': {'Latitude': 30.0074, 'Longitude': 31.4913},
    '6th of October City (Western Desert Boundary)': {'Latitude': 29.9723, 'Longitude': 30.9458},
    'Downtown Cairo (centered near Tahrir Square)': {'Latitude': 30.0444, 'Longitude': 31.2357}
}

START_DATE = "2023-09-01"  # Start date for data retreived
END_DATE = "2026-08-31"    # End date for data retreived
merged_df_list = []        # List of merged lists

In [10]:
# For each district, a dataframe is created with the Air Quality data merged with Weather data
for district_name, district_coordinates in districts_map.items() :

    # print(district_name , district_coordinates)
    # Fetch Air Quality data via Open-Meteo API 
    aq_df = get_aq_data(district_coordinates,START_DATE,END_DATE)

    # Fetch Weather data via Open-Meteo API
    weather_df = get_weather_data(district_coordinates,START_DATE,END_DATE)

    # Merge both dataframes on 'date' column
    df_merged = pd.merge(aq_df, weather_df, on="date", how="inner")
    # Add a Location column to distinguish them after
    df_merged["Location"] = district_name

    # Append to the list of merged dataframes
    merged_df_list.append(df_merged)

Coordinates: 29.800003051757812°N 31.300003051757812°E
Elevation: 61.0 m asl
Timezone difference to GMT+0: 0s
Coordinates: 29.841827392578125°N 31.29069709777832°E
Elevation: 61.0 m asl
Timezone difference to GMT+0: 0s
Coordinates: 30.100000381469727°N 31.200000762939453°E
Elevation: 29.0 m asl
Timezone difference to GMT+0: 0s
Coordinates: 30.123022079467773°N 31.226634979248047°E
Elevation: 29.0 m asl
Timezone difference to GMT+0: 0s
Coordinates: 30.0°N 31.5°E
Elevation: 346.0 m asl
Timezone difference to GMT+0: 0s
Coordinates: 29.982423782348633°N 31.46853256225586°E
Elevation: 346.0 m asl
Timezone difference to GMT+0: 0s
Coordinates: 30.0°N 30.900001525878906°E
Elevation: 184.0 m asl
Timezone difference to GMT+0: 0s
Coordinates: 29.982423782348633°N 30.94405746459961°E
Elevation: 184.0 m asl
Timezone difference to GMT+0: 0s
Coordinates: 30.0°N 31.200000762939453°E
Elevation: 22.0 m asl
Timezone difference to GMT+0: 0s
Coordinates: 30.052722930908203°N 31.19019889831543°E
Elevation: 

In [19]:
for df in merged_df_list:
    print(df.shape)


(26304, 16)
(26304, 16)
(26304, 16)
(26304, 16)
(26304, 16)


In [26]:
merged_df_list[0].to_csv("Helwan (Industrial South).csv")
merged_df_list[0].head()


,date,pm10,pm2_5,uv_index,carbon_monoxide,carbon_dioxide,nitrogen_dioxide,sulphur_dioxide,ozone,temperature_2m,relative_humidity_2m,surface_pressure,precipitation,wind_speed_10m,wind_direction_10m,Location
0,2023-09-01 00:00:00+00:00,16.299999,9.7,0.0,188.0,NaN,12.1,17.900000,62.0,26.049999,73.075142,1005.973572,0.0,16.454008,10.080544,Helwan (Industrial South)
1,2023-09-01 01:00:00+00:00,16.900000,9.6,0.0,186.0,NaN,12.1,17.600000,59.0,25.600000,78.823761,1005.764587,0.0,14.917212,8.325568,Helwan (Industrial South)
2,2023-09-01 02:00:00+00:00,16.400000,9.3,0.0,182.0,NaN,12.2,17.200001,54.0,25.150000,82.458199,1005.753723,0.0,14.440443,4.289077,Helwan (Industrial South)
3,2023-09-01 03:00:00+00:00,16.600000,9.3,0.0,182.0,NaN,12.5,17.400000,52.0,24.750000,84.965622,1005.843689,0.0,14.440443,4.289077,Helwan (Industrial South)
4,2023-09-01 04:00:00+00:00,17.400000,9.8,0.1,187.0,NaN,13.4,18.600000,53.0,24.700001,84.960449,1006.140625,0.0,15.137133,2.726261,Helwan (Industrial South)


In [27]:
merged_df_list[1].to_csv("Shobra El-Kheima (Industrial North).csv")
merged_df_list[1].head()

,date,pm10,pm2_5,uv_index,carbon_monoxide,carbon_dioxide,nitrogen_dioxide,sulphur_dioxide,ozone,temperature_2m,relative_humidity_2m,surface_pressure,precipitation,wind_speed_10m,wind_direction_10m,Location
0,2023-09-01 00:00:00+00:00,30.000000,15.200000,0.0,160.0,NaN,11.7,10.9,68.0,26.100000,77.937965,1009.852356,0.0,10.703569,19.653913,Shobra El-Kheima (Industrial North)
1,2023-09-01 01:00:00+00:00,31.900000,16.700001,0.0,166.0,NaN,11.9,11.7,64.0,25.750000,80.301064,1009.648926,0.0,10.390226,14.036275,Shobra El-Kheima (Industrial North)
2,2023-09-01 02:00:00+00:00,33.299999,18.500000,0.0,172.0,NaN,12.3,11.9,61.0,25.350000,82.482002,1009.644531,0.0,9.826088,8.426887,Shobra El-Kheima (Industrial North)
3,2023-09-01 03:00:00+00:00,33.700001,19.299999,0.0,165.0,NaN,12.2,11.9,62.0,25.450001,81.247017,1009.745361,0.0,11.525623,14.470333,Shobra El-Kheima (Industrial North)
4,2023-09-01 04:00:00+00:00,34.200001,16.200001,0.1,172.0,NaN,12.4,12.1,62.0,25.549999,79.300117,1010.045349,0.0,11.275530,16.699326,Shobra El-Kheima (Industrial North)


In [29]:
merged_df_list[2].to_csv("New Cairo-5th Settlement (Eastern Residential Suburbs).csv")
merged_df_list[2].head()

,date,pm10,pm2_5,uv_index,carbon_monoxide,carbon_dioxide,nitrogen_dioxide,sulphur_dioxide,ozone,temperature_2m,relative_humidity_2m,surface_pressure,precipitation,wind_speed_10m,wind_direction_10m,Location
0,2023-09-01 00:00:00+00:00,31.500000,13.7,0.0,154.0,NaN,9.1,6.1,71.0,23.299999,86.129494,974.415161,0.0,18.278645,32.124966,New Cairo / 5th Settlement (Eastern Residentia...
1,2023-09-01 01:00:00+00:00,31.400000,12.8,0.0,156.0,NaN,10.0,7.2,65.0,22.900000,90.710716,974.074768,0.0,17.786331,35.942207,New Cairo / 5th Settlement (Eastern Residentia...
2,2023-09-01 02:00:00+00:00,32.400002,13.5,0.0,157.0,NaN,9.5,7.8,61.0,22.500000,93.796684,974.118713,0.0,16.485485,31.607454,New Cairo / 5th Settlement (Eastern Residentia...
3,2023-09-01 03:00:00+00:00,24.900000,14.7,0.0,163.0,NaN,9.6,7.8,57.0,22.100000,96.402023,974.162109,0.0,16.746773,25.463305,New Cairo / 5th Settlement (Eastern Residentia...
4,2023-09-01 04:00:00+00:00,33.400002,13.8,0.1,168.0,NaN,10.6,8.3,67.0,22.049999,96.105934,974.443970,0.0,17.072504,24.943882,New Cairo / 5th Settlement (Eastern Residentia...


In [30]:
merged_df_list[3].to_csv("6th of October City (Western Desert Boundary).csv")
merged_df_list[3].head()


,date,pm10,pm2_5,uv_index,carbon_monoxide,carbon_dioxide,nitrogen_dioxide,sulphur_dioxide,ozone,temperature_2m,relative_humidity_2m,surface_pressure,precipitation,wind_speed_10m,wind_direction_10m,Location
0,2023-09-01 00:00:00+00:00,13.4,6.8,0.0,170.0,NaN,5.4,9.2,71.0,24.549999,80.635902,992.562256,0.0,15.463244,12.094739,6th of October City (Western Desert Boundary)
1,2023-09-01 01:00:00+00:00,13.0,6.8,0.0,170.0,NaN,5.3,8.4,68.0,24.250000,83.112068,992.345215,0.0,14.081478,4.398633,6th of October City (Western Desert Boundary)
2,2023-09-01 02:00:00+00:00,12.3,6.7,0.0,169.0,NaN,5.1,7.3,65.0,23.799999,86.442268,992.313599,0.0,14.404499,1.432063,6th of October City (Western Desert Boundary)
3,2023-09-01 03:00:00+00:00,11.9,6.6,0.0,169.0,NaN,5.1,6.8,64.0,23.650000,87.762054,992.303040,0.0,14.471821,5.710507,6th of October City (Western Desert Boundary)
4,2023-09-01 04:00:00+00:00,11.7,6.6,0.1,170.0,NaN,5.6,7.3,66.0,23.950001,85.142784,992.617981,0.0,16.873980,11.070170,6th of October City (Western Desert Boundary)


In [31]:
merged_df_list[4].to_csv("Downtown Cairo.csv")
merged_df_list[4].head()

,date,pm10,pm2_5,uv_index,carbon_monoxide,carbon_dioxide,nitrogen_dioxide,sulphur_dioxide,ozone,temperature_2m,relative_humidity_2m,surface_pressure,precipitation,wind_speed_10m,wind_direction_10m,Location
0,2023-09-01 00:00:00+00:00,29.400000,17.400000,0.0,174.0,NaN,14.600000,15.700000,57.0,25.950001,77.441544,1010.658020,0.0,11.525623,14.470333,Downtown Cairo (centered near Tahrir Square)
1,2023-09-01 01:00:00+00:00,33.900002,18.900000,0.0,172.0,NaN,15.600000,16.100000,54.0,25.600000,80.281265,1010.455261,0.0,11.440978,12.724360,Downtown Cairo (centered near Tahrir Square)
2,2023-09-01 02:00:00+00:00,35.900002,20.400000,0.0,173.0,NaN,15.600000,16.600000,53.0,25.200001,82.464149,1010.451782,0.0,10.495713,5.906055,Downtown Cairo (centered near Tahrir Square)
3,2023-09-01 03:00:00+00:00,38.099998,22.200001,0.0,179.0,NaN,14.600000,17.700001,52.0,25.100000,82.956726,1010.550903,0.0,11.367109,10.954029,Downtown Cairo (centered near Tahrir Square)
4,2023-09-01 04:00:00+00:00,39.099998,16.900000,0.1,185.0,NaN,17.799999,19.500000,52.0,25.299999,80.487183,1010.851746,0.0,12.224107,13.627024,Downtown Cairo (centered near Tahrir Square)


In [23]:
result = pd.concat(merged_df_list, ignore_index=True)

In [24]:
result

,date,pm10,pm2_5,uv_index,carbon_monoxide,carbon_dioxide,nitrogen_dioxide,sulphur_dioxide,ozone,temperature_2m,relative_humidity_2m,surface_pressure,precipitation,wind_speed_10m,wind_direction_10m,Location
0,2023-09-01 00:00:00+00:00,16.299999,9.700000,0.0,188.0,NaN,12.100000,17.900000,62.0,26.049999,73.075142,1005.973572,0.0,16.454008,10.080544,Helwan (Industrial South)
1,2023-09-01 01:00:00+00:00,16.900000,9.600000,0.0,186.0,NaN,12.100000,17.600000,59.0,25.600000,78.823761,1005.764587,0.0,14.917212,8.325568,Helwan (Industrial South)
2,2023-09-01 02:00:00+00:00,16.400000,9.300000,0.0,182.0,NaN,12.200000,17.200001,54.0,25.150000,82.458199,1005.753723,0.0,14.440443,4.289077,Helwan (Industrial South)
3,2023-09-01 03:00:00+00:00,16.600000,9.300000,0.0,182.0,NaN,12.500000,17.400000,52.0,24.750000,84.965622,1005.843689,0.0,14.440443,4.289077,Helwan (Industrial South)
4,2023-09-01 04:00:00+00:00,17.400000,9.800000,0.1,187.0,NaN,13.400000,18.600000,53.0,24.700001,84.960449,1006.140625,0.0,15.137133,2.726261,Helwan (Industrial South)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131515,2026-08-31 19:00:00+00:00,28.100000,18.400000,0.0,223.0,448.0,19.799999,32.599998,84.0,30.400000,43.490379,1007.303406,0.0,14.580000,360.000000,Downtown Cairo (centered near Tahrir Square)
131516,2026-08-31 20:00:00+00:00,30.400000,17.700001,0.0,222.0,447.0,22.600000,28.299999,67.0,29.750000,44.715954,1007.796753,0.0,9.387650,355.601379,Downtown Cairo (centered near Tahrir Square)
131517,2026-08-31 21:00:00+00:00,31.600000,17.200001,0.0,230.0,447.0,29.100000,31.299999,55.0,28.950001,46.829666,1007.889954,0.0,10.172787,346.701416,Downtown Cairo (centered near Tahrir Square)
131518,2026-08-31 22:00:00+00:00,35.000000,18.500000,0.0,231.0,447.0,32.299999,31.500000,51.0,28.000000,49.961582,1008.081543,0.0,9.867218,345.203186,Downtown Cairo (centered near Tahrir Square)


In [25]:
result.to_csv("Merged_AQ_Weather_Data_Meteo.csv")